# Week 10: Design Patterns — Strategy Pattern — PHASE 4: Verifying & Swapping Components

*Object Oriented Programming . 3 Hours . Dr. Arif Solmaz*

---

## Learning Objectives

By the end of this week, you will be able to:

| # | Objective |
|---|----------|
| 1 | Explain what a design pattern is |
| 2 | Describe the problem that the Strategy pattern solves |
| 3 | Implement the Strategy pattern using Python classes |
| 4 | Swap behaviors at runtime without changing existing code |
| 5 | Pass strategy objects as arguments to other objects |
| 6 | Apply the Strategy pattern to a real-world filter selection example |

---

## 🎯 Core Mastery Connection

The Strategy pattern is composition in action — you swap one component for another at runtime. A `SensorReader` does not care whether it uses a `MovingAverageFilter` or a `MedianFilter`; it only cares that the component fulfills the strategy contract. This is the ultimate payoff of building systems from composable, interchangeable parts.

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import unittest

## Part 1: What is a Design Pattern?

In engineering, we often encounter the **same types of problems** again and again. Over time, experienced programmers discovered **common solutions** that work well. These solutions are called **design patterns**.

A design pattern is **not** a piece of code you copy-paste. It is an **idea** — a template for solving a specific type of problem.

### Analogy from engineering

Think of a gear system. Engineers don't reinvent gears every time they need to transmit motion. They pick from known gear arrangements (spur, bevel, worm) depending on the situation. Design patterns are the same idea, but for software.

| Term | Meaning |
|------|--------|
| Design Pattern | A reusable solution template for a common problem |
| Strategy Pattern | A pattern that lets you swap algorithms/behaviors at runtime |
| Gang of Four (GoF) | The four authors who cataloged 23 classic patterns |

> **Key Idea:** Patterns help you write code that is flexible and easy to change.

---

## Part 2: The Problem — Hard-Coded Behavior

Imagine you have a sensor that gives noisy readings. You want to **filter** the noise. You start with a simple moving average.

But later, your supervisor says: "Use a median filter instead, it works better for spikes."

If the filtering logic is **hard-coded** inside your class, you have to change the class every time you want a different filter. This is a problem.

**Figure 10.1** — The hard-coded approach (bad):

```
SensorReader
    └── filter_data()   <-- contains the algorithm directly
                           changing the algorithm = changing this class
```

Let's see this problem in code:

In [ ]:
# BAD APPROACH: Hard-coded filtering logic

class SensorReaderV1:
    """Reads sensor data and filters it. Filter is hard-coded."""
    def __init__(self):
        self.readings = []

    def add_reading(self, value):
        self.readings.append(value)

    def get_filtered_value(self, window_size=3):
        """Hard-coded moving average filter."""
        if len(self.readings) < window_size:
            return self.readings[-1] if self.readings else 0
        window = self.readings[-window_size:]
        return sum(window) / len(window)


# This works, but...
reader = SensorReaderV1()
for val in [10, 12, 100, 11, 13]:  # 100 is a spike/noise
    reader.add_reading(val)

print(f"Moving average result: {reader.get_filtered_value(3):.1f}")
# The spike (100) pulls the average up. A median filter would handle this better.
# But to change the filter, we must MODIFY the class!

**What's wrong with this?**

- To use a different filter, you must **edit** the `SensorReaderV1` class
- You cannot switch filters at runtime
- Every change risks breaking existing code

---

## Part 3: The Strategy Pattern Idea

The **Strategy Pattern** says:

> **Take the part that changes (the algorithm) and put it in a separate class. Then pass that class to the object that needs it.**

**Figure 10.2** — The Strategy pattern approach (good):

```
SensorReader
    └── uses a FilterStrategy object

FilterStrategy (interface)
    ├── MovingAverageFilter
    ├── MedianFilter
    └── AnyNewFilter (can add without changing SensorReader!)
```

### The three roles:

| Role | What it does | Example |
|------|-------------|--------|
| **Context** | The class that uses a strategy | `SensorReader` |
| **Strategy** | The interface (common method name) | `FilterStrategy` with `apply()` |
| **Concrete Strategies** | The actual implementations | `MovingAverageFilter`, `MedianFilter` |

---

## Part 4: Implementing Strategy with Classes

Let's implement the Strategy pattern step by step.

**Step 1:** Create the strategy classes (each with the same method name).

**Step 2:** Create the context class that **accepts** a strategy object.

**Step 3:** Use the strategy by calling its method.

**Figure 10.1** — Concrete strategy classes with a shared apply() method

In [ ]:
# Step 1: Define strategy classes
# Each strategy class has an apply(data, window_size) method

class MovingAverageFilter:
    """Calculates the average of the last N values."""
    def apply(self, data, window_size):
        if len(data) < window_size:
            return data[-1] if data else 0
        window = data[-window_size:]
        return sum(window) / len(window)


class MedianFilter:
    """Returns the median of the last N values."""
    def apply(self, data, window_size):
        if len(data) < window_size:
            return data[-1] if data else 0
        window = sorted(data[-window_size:])
        mid = len(window) // 2
        return window[mid]


# Let's test the strategies alone
data = [10, 12, 100, 11, 13]

avg_filter = MovingAverageFilter()
med_filter = MedianFilter()

print(f"Moving Average (last 3): {avg_filter.apply(data, 3):.1f}")
print(f"Median Filter  (last 3): {med_filter.apply(data, 3):.1f}")

**Figure 10.2** — Context class (SensorReader) that delegates to a strategy object

In [ ]:
# Step 2: Create the context class that uses a strategy

class SensorReader:
    """Reads sensor data and filters it using a strategy object."""
    def __init__(self, filter_strategy):
        self.readings = []
        self.filter_strategy = filter_strategy  # Store the strategy object

    def add_reading(self, value):
        self.readings.append(value)

    def get_filtered_value(self, window_size=3):
        # Delegate filtering to the strategy object
        return self.filter_strategy.apply(self.readings, window_size)

    def set_filter(self, filter_strategy):
        """Change the filter strategy at runtime."""
        self.filter_strategy = filter_strategy


# Step 3: Use it!
# Create a sensor reader with moving average filter
reader = SensorReader(MovingAverageFilter())

for val in [10, 12, 100, 11, 13]:
    reader.add_reading(val)

print(f"With Moving Average: {reader.get_filtered_value(3):.1f}")

# Switch to median filter at runtime!
reader.set_filter(MedianFilter())
print(f"With Median Filter:  {reader.get_filtered_value(3):.1f}")

Notice that:
- We **never changed** the `SensorReader` class
- We simply **swapped** the strategy object
- The `SensorReader` does not know (or care) which filter it is using

---

## Part 5: Swapping Strategies at Runtime

One of the biggest benefits of the Strategy pattern is that you can **change behavior while the program is running**.

**Figure 10.3** — Runtime strategy swapping:

```
Time 0:  SensorReader  --uses-->  MovingAverageFilter
Time 1:  SensorReader  --uses-->  MedianFilter          (swapped!)
Time 2:  SensorReader  --uses-->  MovingAverageFilter    (swapped back!)
```

The `SensorReader` class stays the same. Only the strategy object changes.

In [ ]:
# Let's add a third strategy: Last Value (no filtering)

class LastValueFilter:
    """Simply returns the last reading (no filtering)."""
    def apply(self, data, window_size):
        return data[-1] if data else 0


# Simulate a scenario where we switch filters
reader = SensorReader(MovingAverageFilter())

sensor_data = [10, 12, 100, 11, 13, 14, 200, 15, 16]

for val in sensor_data:
    reader.add_reading(val)

# Compare all three strategies on the same data
strategies = [
    ("Moving Average", MovingAverageFilter()),
    ("Median", MedianFilter()),
    ("Last Value", LastValueFilter()),
]

print("Comparing filter strategies on noisy sensor data:")
print(f"Raw data: {sensor_data}")
print()

for name, strategy in strategies:
    reader.set_filter(strategy)
    result = reader.get_filtered_value(3)
    print(f"{name:20s}: {result:.1f}")

### Adding a new strategy is easy!

To add a new filter, you only need to:
1. Create a new class with an `apply()` method
2. Pass it to the `SensorReader`

You do **not** need to change `SensorReader` at all. This follows the **Open/Closed Principle**: open for extension, closed for modification.

---

## Part 6: Real-World Example — Filter Selection

Let's build a more complete example. Imagine a vibration monitoring system on a CNC machine. The system reads vibration data and applies different filters depending on the operating mode.

| Mode | Filter | Why |
|------|--------|----|
| Normal operation | Moving Average | Smooth, general-purpose |
| Spike detection | Median | Ignores occasional spikes |
| Raw monitoring | Last Value | No filtering needed |

**Figure 10.4** — CNC vibration monitoring system:

```
Vibration Sensor  -->  VibrationMonitor  --uses-->  FilterStrategy
                            |                            |
                       set_mode()              MovingAverageFilter
                                               MedianFilter
                                               LastValueFilter
```

In [ ]:
# Complete real-world example: CNC vibration monitor

class VibrationMonitor:
    """Monitors vibration on a CNC machine."""

    # Define available modes and their strategies
    MODES = {
        "normal": MovingAverageFilter(),
        "spike_detection": MedianFilter(),
        "raw": LastValueFilter(),
    }

    def __init__(self, mode="normal"):
        self.readings = []
        self.set_mode(mode)

    def set_mode(self, mode):
        """Switch the filtering mode."""
        if mode not in self.MODES:
            raise ValueError(f"Unknown mode: {mode}")
        self.mode = mode
        self.filter_strategy = self.MODES[mode]

    def add_reading(self, value):
        self.readings.append(value)

    def current_value(self, window_size=5):
        return self.filter_strategy.apply(self.readings, window_size)

    def is_above_threshold(self, threshold, window_size=5):
        """Check if filtered vibration exceeds safety threshold."""
        return self.current_value(window_size) > threshold


# Simulate vibration data with a spike
vibration_data = [2.1, 2.3, 2.0, 50.0, 2.2, 2.1, 2.4, 2.3, 2.5]

monitor = VibrationMonitor(mode="normal")
for v in vibration_data:
    monitor.add_reading(v)

print("CNC Vibration Monitor")
print(f"Raw data: {vibration_data}")
print(f"Safety threshold: 5.0 mm/s")
print()

for mode_name in ["normal", "spike_detection", "raw"]:
    monitor.set_mode(mode_name)
    value = monitor.current_value(5)
    alarm = "ALARM!" if monitor.is_above_threshold(5.0, 5) else "OK"
    print(f"Mode: {mode_name:20s} | Value: {value:6.2f} | Status: {alarm}")

Notice how the median filter correctly ignores the spike (50.0), while the moving average is still affected by it. The choice of strategy matters!

---

## Part 7: Benefits of Strategy Pattern

Let's summarize why the Strategy pattern is useful:

| Benefit | Explanation |
|---------|------------|
| **Flexibility** | Change behavior without changing the class |
| **Open/Closed** | Add new strategies without modifying existing code |
| **Testability** | Test each strategy independently |
| **Readability** | Each strategy is a small, focused class |
| **Runtime swap** | Change behavior while the program is running |

### When to use the Strategy pattern:

- You have multiple ways to do the same thing (different algorithms)
- You want to switch between them easily
- You want to add new options without changing existing code

### When NOT to use it:

- You only have one algorithm and it will never change
- The extra classes make simple code unnecessarily complicated

**Figure 10.5** — Strategy pattern summary:

```
WITHOUT Strategy:              WITH Strategy:

class Robot:                   class Robot:
    if mode == "A":                strategy.execute()
        do_A()
    elif mode == "B":          class StrategyA:
        do_B()                     execute()
    elif mode == "C":          class StrategyB:
        do_C()                     execute()
    # ... more elif ...        class StrategyC:
                                   execute()
```

---

### Passing strategy objects as arguments

Strategy objects can also be passed to **functions**, not just stored in classes. This is useful for one-time operations.

**Figure 10.3** — Passing a strategy object to a standalone function

In [ ]:
# Passing a strategy object to a function

def process_sensor_data(data, filter_strategy, window_size=3):
    """Process sensor data using a given filter strategy."""
    results = []
    for i in range(len(data)):
        subset = data[:i+1]  # Data up to current point
        filtered = filter_strategy.apply(subset, window_size)
        results.append(filtered)
    return results


# Same data, different strategies passed as arguments
raw_data = [5, 6, 50, 7, 8, 6, 5]

avg_results = process_sensor_data(raw_data, MovingAverageFilter())
med_results = process_sensor_data(raw_data, MedianFilter())

print("Step-by-step filtering comparison:")
print(f"{'Raw':>6} {'Average':>10} {'Median':>10}")
print("-" * 28)
for raw, avg, med in zip(raw_data, avg_results, med_results):
    print(f"{raw:>6} {avg:>10.1f} {med:>10.1f}")

---

## Exercises

> **Composition lens:** Every strategy you implement is a swappable component. The exercises below ask you to build components that can be plugged into a context object interchangeably — this is the core mastery in its purest form.

Complete the following exercises in the code cells below.

### Exercise 1: Implement a Min Filter Strategy (Easy)

Create a `MinFilter` class that returns the **minimum** value from the last N readings. It should have an `apply(data, window_size)` method. Test it with the `SensorReader` class.

<details>
<summary>💡 Hint</summary>

Use the `min()` function on `data[-window_size:]`. Follow the same structure as `MovingAverageFilter`.
</details>

In [ ]:
# ✏️ [EX1] Implement MinFilter strategy and test it


### Exercise 2: Implement a Max Filter Strategy (Easy)

Create a `MaxFilter` class that returns the **maximum** value from the last N readings. Test it alongside the other filters.

<details>
<summary>💡 Hint</summary>

Same structure as MinFilter but use `max()` instead of `min()`.
</details>

In [ ]:
# ✏️ [EX2] Implement MaxFilter strategy and test it


### Exercise 3: Speed Control Strategy (Easy)

Create a `MotorController` class that uses a speed calculation strategy. Implement two strategies:
- `LinearSpeed` — speed increases linearly with input: `speed = input_value * factor`
- `QuadraticSpeed` — speed increases quadratically: `speed = input_value ** 2 * factor`

Both strategies should have a `calculate(input_value)` method.

<details>
<summary>💡 Hint</summary>

Create strategy classes with `__init__(self, factor)` and `calculate(self, input_value)`. The `MotorController` accepts a strategy in its constructor.
</details>

In [ ]:
# ✏️ [EX3] Motor controller with speed calculation strategies


### Exercise 4: Sorting Strategy (Easy)

Create a `DataProcessor` class that sorts a list of numbers. Implement two sorting strategies:
- `AscendingSort` — sorts smallest to largest
- `DescendingSort` — sorts largest to smallest

Both should have a `sort(data)` method that returns a new sorted list.

<details>
<summary>💡 Hint</summary>

Use Python's `sorted()` function. For descending, use `sorted(data, reverse=True)`.
</details>

In [ ]:
# ✏️ [EX4] Data processor with sorting strategies


### Exercise 5: Swap Strategy at Runtime (Medium)

Using the `SensorReader` class from earlier, create a sensor reader that starts with `MovingAverageFilter`, adds 5 readings, prints the filtered value, then swaps to `MedianFilter` and prints the filtered value again.

<details>
<summary>💡 Hint</summary>

Use `reader.set_filter(MedianFilter())` to swap. The readings stay the same; only the filter changes.
</details>

In [ ]:
# ✏️ [EX5] Swap strategies at runtime


### Exercise 6: Unit Conversion Strategy (Medium)

Create a `Measurement` class that stores a value in meters. Implement strategies for displaying in different units:
- `MetersDisplay` — returns value as is
- `CentimetersDisplay` — multiplies by 100
- `MillimetersDisplay` — multiplies by 1000

Each strategy should have a `convert(value)` method.

<details>
<summary>💡 Hint</summary>

The `Measurement` class stores the value in meters and has a `display(strategy)` method that uses the strategy to convert.
</details>

In [ ]:
# ✏️ [EX6] Measurement class with unit conversion strategies


### Exercise 7: Movement Strategy for a Robot (Medium)

Create a `Robot` class with a position `(x, y)`. Implement movement strategies:
- `MoveForward` — increases y by a step
- `MoveRight` — increases x by a step
- `MoveDiagonal` — increases both x and y by a step

Each strategy has a `move(x, y, step)` method that returns a new `(x, y)` tuple.

<details>
<summary>💡 Hint</summary>

The Robot stores its position and has a `move(step)` method that delegates to the current strategy.
</details>

In [ ]:
# ✏️ [EX7] Robot with movement strategies


### Exercise 8: Test Your Strategies (Medium)

Using `unittest` (from last week!), write tests for the `MovingAverageFilter` and `MedianFilter` classes. Write at least 3 tests for each.

<details>
<summary>💡 Hint</summary>

Test with known data. For example, median of [1, 3, 2] is 2. Moving average of [10, 20, 30] with window 3 is 20.0.
</details>

In [ ]:
# ✏️ [EX8] Write unittest tests for filter strategies
import unittest


### Exercise 9: Alarm Strategy (Medium)

Create an `AlarmSystem` class for a factory. Implement alarm strategies:
- `SilentAlarm` — returns `"LOG: {message}"` (just logs it)
- `LoudAlarm` — returns `"ALARM! {message}"` (sounds alarm)
- `EmergencyAlarm` — returns `"EMERGENCY SHUTDOWN: {message}"`

Each strategy has a `trigger(message)` method. The `AlarmSystem` can switch between strategies.

<details>
<summary>💡 Hint</summary>

Use string formatting: `return f"ALARM! {message}"`. The AlarmSystem stores a strategy and has a `raise_alarm(message)` method.
</details>

In [ ]:
# ✏️ [EX9] Alarm system with different alarm strategies


### Exercise 10: Weighted Average Filter (Challenge)

Create a `WeightedAverageFilter` strategy where more recent readings get higher weights. For a window of 3, use weights [1, 2, 3] (oldest to newest). The weighted average is: `sum(value * weight) / sum(weights)`.

Test it with the `SensorReader` class.

<details>
<summary>💡 Hint</summary>

Generate weights using `range(1, window_size + 1)`. Use `zip()` to pair weights with values.
</details>

In [ ]:
# ✏️ [EX10] Implement WeightedAverageFilter strategy


### Exercise 11: Strategy with State (Challenge)

Create an `ExponentialSmoothingFilter` strategy. This filter keeps a running smoothed value using the formula:

`smoothed = alpha * new_value + (1 - alpha) * previous_smoothed`

where `alpha` is a smoothing factor between 0 and 1 (e.g., 0.3). The strategy stores its own state.

<details>
<summary>💡 Hint</summary>

Initialize `smoothed = None` in `__init__`. On first call, set `smoothed = data[-1]`. On subsequent calls, apply the formula. Note: this strategy uses the full data list but only the last value matters each time.
</details>

In [ ]:
# ✏️ [EX11] Implement ExponentialSmoothingFilter strategy


### Exercise 12 (Studio): Make Filter Runtime-Selectable

Build a complete `SignalProcessor` system that:
1. Stores raw signal data (a list of numbers)
2. Has a dictionary of available filter strategies
3. Lets the user select a filter by name (string)
4. Processes all data through the selected filter
5. Can switch filters and reprocess

Include at least 3 filter strategies. Print a comparison table showing the output of each filter on the same data.

<details>
<summary>💡 Hint</summary>

Use a dictionary like `{"average": MovingAverageFilter(), "median": MedianFilter(), ...}`. Add a `select_filter(name)` method that looks up the strategy by name.
</details>

In [ ]:
# ✏️ [EX12] Studio: Build a complete SignalProcessor with runtime-selectable filters


---

## 🌉 Bridge to Next Week

The Strategy pattern is one of the most useful design patterns. You have learned how to separate the **what** (the context) from the **how** (the strategy).

But as your projects grow, you will have many classes spread across many files. How do you **organize** all of this code so it stays manageable?

**Next week**, we will learn about **Modules, Packages, and Import Structure** — Python's system for splitting code into separate files and folders, and importing only what you need. You will learn how to turn your growing toolkit into a well-organized package.

```python
# Sneak peek — Week 11
from sensors.filters import MedianFilter, MovingAverageFilter
from sensors.reader import SensorReader
```

---

## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_10"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")